In [2]:
import requests
import pandas as pd

BASE_URL = "https://api.jolpi.ca/ergast/f1"

# 1. Fetch the complete 2024 Race Schedule (Circuit, Location, Date, Time)
schedule_url = f"{BASE_URL}/2024.json?limit=100"
schedule_response = requests.get(schedule_url).json()
races_list = schedule_response['MRData']['RaceTable']['Races']

# 2. Fetch the 2024 Race Results to get the Race Winner for each round
results_url = f"{BASE_URL}/2024/results.json?limit=1000"
results_response = requests.get(results_url).json()
results_races = results_response['MRData']['RaceTable']['Races']

# Create a lookup dictionary for P1 (Winner) per round
winners_by_round = {}
for race in results_races:
    round_num = race['round']
    for result in race.get('Results', []):
        if result.get('position') == '1':
            winner_name = f"{result['Driver']['givenName']} {result['Driver']['familyName']}"
            team = result['Constructor']['name']
            winners_by_round[round_num] = f"{winner_name} ({team})"

# 3. Combine Schedule and Results into a structured dataset
all_races = []

for race in races_list:
    round_num = race['round']
    all_races.append({
        'Round': int(round_num),
        'Grand Prix': race['raceName'],
        'Circuit': race['Circuit']['circuitName'],
        'Locality': race['Circuit']['Location']['locality'],
        'Country': race['Circuit']['Location']['country'],
        'Date': race['date'],
        'Winner': winners_by_round.get(round_num, 'N/A')
    })

# 4. Convert to DataFrame
df_races = pd.DataFrame(all_races)

# Display all 24 races of the 2024 season
pd.set_option('display.max_rows', 30)
df_races

,Round,Grand Prix,Circuit,Locality,Country,Date,Winner
0,1,Bahrain Grand Prix,Bahrain International Circuit,Sakhir,Bahrain,2024-03-02,Max Verstappen (Red Bull)
1,2,Saudi Arabian Grand Prix,Jeddah Corniche Circuit,Jeddah,Saudi Arabia,2024-03-09,Max Verstappen (Red Bull)
2,3,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,2024-03-24,Carlos Sainz (Ferrari)
3,4,Japanese Grand Prix,Suzuka Circuit,Suzuka,Japan,2024-04-07,Max Verstappen (Red Bull)
4,5,Chinese Grand Prix,Shanghai International Circuit,Shanghai,China,2024-04-21,Max Verstappen (Red Bull)
5,6,Miami Grand Prix,Miami International Autodrome,Miami,USA,2024-05-05,Lando Norris (McLaren)
6,7,Emilia Romagna Grand Prix,Autodromo Enzo e Dino Ferrari,Imola,Italy,2024-05-19,N/A
7,8,Monaco Grand Prix,Circuit de Monaco,Monte Carlo,Monaco,2024-05-26,N/A
8,9,Canadian Grand Prix,Circuit Gilles Villeneuve,Montreal,Canada,2024-06-09,N/A
9,10,Spanish Grand Prix,Circuit de Barcelona-Catalunya,Barcelona,Spain,2024-06-23,N/A
